In [1]:
# [MERGE 1] Load main + modern full datasets and align schemas

from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path(r"C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset")

main_path = BASE_DIR / "data" / "final" / "publication" / "idiomx_full.parquet"
modern_path = BASE_DIR / "data" / "final" / "publication" / "idiomx_modern_full.parquet"

output_parquet = BASE_DIR / "data" / "final" / "publication" / "idiomx_extended_full.parquet"
output_csv = BASE_DIR / "data" / "final" / "publication" / "csv" / "idiomx_extended_full.csv"

df_main = pd.read_parquet(main_path)
df_modern = pd.read_parquet(modern_path)

print("Main shape  :", df_main.shape)
print("Modern shape:", df_modern.shape)

Main shape  : (174956, 52)
Modern shape: (14524, 63)


In [2]:
# [MERGE 2] Add source_dataset if missing and inspect schema differences

if "source_dataset" not in df_main.columns:
    df_main["source_dataset"] = "idiomx_main"

if "source_dataset" not in df_modern.columns:
    df_modern["source_dataset"] = "idiomx_modern"

main_cols = set(df_main.columns)
modern_cols = set(df_modern.columns)

only_in_main = sorted(main_cols - modern_cols)
only_in_modern = sorted(modern_cols - main_cols)

print("Columns only in main dataset:")
print(only_in_main)

print("\nColumns only in modern dataset:")
print(only_in_modern)

Columns only in main dataset:
[]

Columns only in modern dataset:
['explanation_fr', 'idiom_canonical_meaning_french', 'idiom_in_example_french', 'idiom_in_example_meaning_french', 'idiom_level_explanation_fr', 'idiom_validity_label', 'meaning_paraphrases_fr', 'offensive_flag', 'regionality', 'slang_strength']


In [3]:
# [MERGE 3] Align both datasets to the union of columns

all_cols = sorted(main_cols | modern_cols)

for col in all_cols:
    if col not in df_main.columns:
        df_main[col] = pd.NA
    if col not in df_modern.columns:
        df_modern[col] = pd.NA

df_main = df_main[all_cols].copy()
df_modern = df_modern[all_cols].copy()

print("Aligned main shape  :", df_main.shape)
print("Aligned modern shape:", df_modern.shape)
print("Total unified columns:", len(all_cols))

Aligned main shape  : (174956, 63)
Aligned modern shape: (14524, 63)
Total unified columns: 63


In [4]:
# [MERGE 4] Concatenate both full datasets

df_extended_full = pd.concat([df_main, df_modern], ignore_index=True)

print("Merged full shape:", df_extended_full.shape)
print("\nSource dataset distribution:")
print(df_extended_full["source_dataset"].value_counts(dropna=False))

Merged full shape: (189480, 63)

Source dataset distribution:
source_dataset
idiomx_main         174956
modern_extension     14524
Name: count, dtype: int64


In [5]:
# [MERGE 5 - OPTIONAL] Remove exact duplicate example rows only

dedup_keys = [
    "idiom_canonical",
    "example",
    "example_usage_label",
    "row_type",
    "context_type"
]

existing_dedup_keys = [c for c in dedup_keys if c in df_extended_full.columns]

before = len(df_extended_full)
df_extended_full = df_extended_full.drop_duplicates(subset=existing_dedup_keys).copy()
after = len(df_extended_full)

print(f"Rows before dedup: {before}")
print(f"Rows after dedup : {after}")
print(f"Removed          : {before - after}")

Rows before dedup: 189480
Rows after dedup : 189455
Removed          : 25


In [6]:
# [MERGE 6] Reorder columns for cleaner final export

preferred_order = [
    "idiom_id",
    "idiom_canonical",
    "idiom_surface",
    "example",
    "example_usage_label",
    "is_example_idiom",

    "idiom_canonical_meaning",
    "idiom_canonical_meaning_arabic",
    "idiom_canonical_meaning_french",

    "idiom_in_example_meaning_en",
    "idiom_in_example_meaning_arabic",
    "idiom_in_example_meaning_french",

    "example_raw",
    "example_normalized",
    "example_language",
    "meaning_language",

    "source",
    "source_type",
    "source_url",
    "record_origin",
    "license_source",
    "source_dataset",

    "pos",
    "tags",
    "idiom_confidence",

    "is_idiom",
    "idiom_validity_label",
    "ambiguity_flag",
    "idiom_compositionality_level",
    "idiom_register",
    "idiom_domain",
    "learner_difficulty",
    "slang_strength",
    "regionality",
    "offensive_flag",

    "idiom_in_example_arabic",
    "idiom_in_example_french",

    "idiom_level_explanation_en",
    "idiom_level_explanation_ar",
    "idiom_level_explanation_fr",

    "explanation_en",
    "explanation_ar",
    "explanation_fr",

    "hard_negative_idioms",
    "meaning_paraphrases_en",
    "meaning_paraphrases_ar",
    "meaning_paraphrases_fr",

    "is_generated_example",
    "enrichment_model",
    "enrichment_version",
    "validation_status",

    "context_type",
    "source_style",
    "minimal_pair_id",
    "paraphrase_group_id",

    "is_adversarial_example",
    "adversarial_type",
    "expected_label",
    "row_type",

    "sentence_length_chars",
    "sentence_length_words",
    "semantic_similarity_example_vs_meaning",
    "semantic_quality",
]

first_cols = [c for c in preferred_order if c in df_extended_full.columns]
remaining_cols = [c for c in df_extended_full.columns if c not in first_cols]

df_extended_full = df_extended_full[first_cols + remaining_cols].copy()

print("Final merged columns:")
print(df_extended_full.columns.tolist())
print("\nFinal merged shape:", df_extended_full.shape)

Final merged columns:
['idiom_id', 'idiom_canonical', 'idiom_surface', 'example', 'example_usage_label', 'is_example_idiom', 'idiom_canonical_meaning', 'idiom_canonical_meaning_arabic', 'idiom_canonical_meaning_french', 'idiom_in_example_meaning_en', 'idiom_in_example_meaning_arabic', 'idiom_in_example_meaning_french', 'example_raw', 'example_normalized', 'example_language', 'meaning_language', 'source', 'source_type', 'source_url', 'record_origin', 'license_source', 'source_dataset', 'pos', 'tags', 'idiom_confidence', 'is_idiom', 'idiom_validity_label', 'ambiguity_flag', 'idiom_compositionality_level', 'idiom_register', 'idiom_domain', 'learner_difficulty', 'slang_strength', 'regionality', 'offensive_flag', 'idiom_in_example_arabic', 'idiom_in_example_french', 'idiom_level_explanation_en', 'idiom_level_explanation_ar', 'idiom_level_explanation_fr', 'explanation_en', 'explanation_ar', 'explanation_fr', 'hard_negative_idioms', 'meaning_paraphrases_en', 'meaning_paraphrases_ar', 'meaning

In [7]:
# [MERGE 7] Save merged dataset

output_parquet.parent.mkdir(parents=True, exist_ok=True)
output_csv.parent.mkdir(parents=True, exist_ok=True)

df_extended_full.to_parquet(output_parquet, index=False)
df_extended_full.to_csv(output_csv, index=False, encoding="utf-8-sig")

print("Saved parquet:", output_parquet)
print("Saved csv    :", output_csv)

Saved parquet: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\data\final\publication\idiomx_extended_full.parquet
Saved csv    : C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\data\final\publication\csv\idiomx_extended_full.csv


In [8]:
# [MERGE 8] Compute and fill derived text-quality columns

import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def normalize_example_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).strip().lower()

    # normalize spaces
    text = re.sub(r"\s+", " ", text)

    # normalize quotes/apostrophes
    text = text.replace("’", "'").replace("“", '"').replace("”", '"')

    # remove extra surrounding punctuation spacing
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)

    return text


def count_words(text: str) -> int:
    if pd.isna(text):
        return 0
    text = str(text).strip()
    if not text:
        return 0
    return len(re.findall(r"\S+", text))


def compute_tfidf_similarity_batch(examples, meanings):
    """
    Lightweight semantic similarity using TF-IDF cosine similarity.
    Good enough as a reproducible fallback for dataset-level scoring.
    """
    sims = []

    for ex, mean in zip(examples, meanings):
        ex = "" if pd.isna(ex) else str(ex).strip()
        mean = "" if pd.isna(mean) else str(mean).strip()

        if not ex or not mean:
            sims.append(np.nan)
            continue

        try:
            vect = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
            X = vect.fit_transform([ex, mean])
            sim = cosine_similarity(X[0:1], X[1:2])[0][0]
            sims.append(float(round(sim, 4)))
        except Exception:
            sims.append(np.nan)

    return sims


def semantic_quality_from_score(score):
    if pd.isna(score):
        return pd.NA
    if score >= 0.35:
        return "high"
    elif score >= 0.18:
        return "medium"
    else:
        return "low"


# --------------------------------------------------
# Ensure columns exist
# --------------------------------------------------
target_cols = [
    "example_normalized",
    "sentence_length_chars",
    "sentence_length_words",
    "semantic_similarity_example_vs_meaning",
    "semantic_quality",
]

for col in target_cols:
    if col not in df_extended_full.columns:
        df_extended_full[col] = pd.NA

# --------------------------------------------------
# Fill example_normalized
# --------------------------------------------------
mask_norm = df_extended_full["example_normalized"].isna() | (df_extended_full["example_normalized"].astype(str).str.strip() == "")
df_extended_full.loc[mask_norm, "example_normalized"] = (
    df_extended_full.loc[mask_norm, "example"].apply(normalize_example_text)
)

# --------------------------------------------------
# Fill sentence length columns
# --------------------------------------------------
mask_chars = df_extended_full["sentence_length_chars"].isna()
df_extended_full.loc[mask_chars, "sentence_length_chars"] = (
    df_extended_full.loc[mask_chars, "example_normalized"].fillna("").astype(str).str.len()
)

mask_words = df_extended_full["sentence_length_words"].isna()
df_extended_full.loc[mask_words, "sentence_length_words"] = (
    df_extended_full.loc[mask_words, "example_normalized"].apply(count_words)
)

# --------------------------------------------------
# Fill semantic similarity
# Uses:
#   example  vs idiom_in_example_meaning_en
# --------------------------------------------------
mask_sim = df_extended_full["semantic_similarity_example_vs_meaning"].isna()

examples = df_extended_full.loc[mask_sim, "example"].tolist()
meanings = df_extended_full.loc[mask_sim, "idiom_in_example_meaning_en"].tolist()

df_extended_full.loc[mask_sim, "semantic_similarity_example_vs_meaning"] = compute_tfidf_similarity_batch(
    examples,
    meanings
)

# --------------------------------------------------
# Fill semantic_quality from similarity
# --------------------------------------------------
mask_quality = df_extended_full["semantic_quality"].isna() | (df_extended_full["semantic_quality"].astype(str).str.strip() == "")
df_extended_full.loc[mask_quality, "semantic_quality"] = (
    df_extended_full.loc[mask_quality, "semantic_similarity_example_vs_meaning"]
    .apply(semantic_quality_from_score)
)

# --------------------------------------------------
# Optional dtype cleanup
# --------------------------------------------------
df_extended_full["sentence_length_chars"] = pd.to_numeric(
    df_extended_full["sentence_length_chars"], errors="coerce"
).astype("Int64")

df_extended_full["sentence_length_words"] = pd.to_numeric(
    df_extended_full["sentence_length_words"], errors="coerce"
).astype("Int64")

df_extended_full["semantic_similarity_example_vs_meaning"] = pd.to_numeric(
    df_extended_full["semantic_similarity_example_vs_meaning"], errors="coerce"
)

print("Derived columns updated.\n")
print(df_extended_full[
    [
        "example",
        "example_normalized",
        "sentence_length_chars",
        "sentence_length_words",
        "idiom_in_example_meaning_en",
        "semantic_similarity_example_vs_meaning",
        "semantic_quality",
    ]
].head(10))

Derived columns updated.

                                             example  \
0  'ark at 'ee, you really think that plan will w...   
1  Can you 'ark at 'ee to check if the noise is c...   
2  As the storm worsened, Togget shouted, ‘’Ark a...   
3  The owl turned its head sharply, and the boy s...   
4  In the meeting, he remarked, ‘’Ark at ’ee, how...   
5  Please ’ark at ’ee the microchip here to obser...   
6  Just saw him quitting the job, ’ark at ’ee, ca...   
7  Can you ’ark at ’ee how clear the sound is fro...   
8  ’Ark at ’ee! What do you mean by quitting so s...   
9  Can you ’ark at ’ee the diagram to find the au...   

                                  example_normalized  sentence_length_chars  \
0     ark at ee you really think that plan will work                     50   
1  can you ark at ee to check if the noise is com...                     68   
2  as the storm worsened togget shouted ark at ee...                     83   
3  the owl turned its head sharply and th

C:\Users\ayman\AppData\Local\Temp\ipykernel_137700\1003336152.py:105: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_extended_full.loc[mask_words, "sentence_length_words"] = (


In [9]:
# [MERGE 10] Train/test split by idiom_canonical and save as parquet

from pathlib import Path
from sklearn.model_selection import train_test_split

BASE_DIR = Path(r"C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset")

publication_dir = BASE_DIR / "data" / "final" / "publication"
publication_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# Split by idiom to avoid leakage
# --------------------------------------------------
unique_idioms = (
    df_extended_full["idiom_canonical"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

train_idioms, test_idioms = train_test_split(
    unique_idioms,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

train_idiom_set = set(train_idioms)
test_idiom_set = set(test_idioms)

df_extended_train = df_extended_full[
    df_extended_full["idiom_canonical"].astype(str).str.strip().isin(train_idiom_set)
].copy()

df_extended_test = df_extended_full[
    df_extended_full["idiom_canonical"].astype(str).str.strip().isin(test_idiom_set)
].copy()

print("Extended full shape :", df_extended_full.shape)
print("Extended train shape:", df_extended_train.shape)
print("Extended test shape :", df_extended_test.shape)

print("\nUnique idioms:")
print("Train:", df_extended_train["idiom_canonical"].nunique())
print("Test :", df_extended_test["idiom_canonical"].nunique())

# safety check
overlap = set(df_extended_train["idiom_canonical"].dropna().astype(str).str.strip()) & \
          set(df_extended_test["idiom_canonical"].dropna().astype(str).str.strip())

print("\nOverlap idioms between train/test:", len(overlap))

Extended full shape : (189455, 63)
Extended train shape: (151527, 63)
Extended test shape : (37600, 63)

Unique idioms:
Train: 11243
Test : 2811

Overlap idioms between train/test: 0


In [10]:
# [MERGE 11] Save full + split parquet files

extended_full_parquet = publication_dir / "idiomx_extended_full.parquet"
extended_train_parquet = publication_dir / "idiomx_extended_train.parquet"
extended_test_parquet = publication_dir / "idiomx_extended_test.parquet"

df_extended_full.to_parquet(extended_full_parquet, index=False)
df_extended_train.to_parquet(extended_train_parquet, index=False)
df_extended_test.to_parquet(extended_test_parquet, index=False)

print("Saved parquet files:")
print("-", extended_full_parquet)
print("-", extended_train_parquet)
print("-", extended_test_parquet)

Saved parquet files:
- C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\data\final\publication\idiomx_extended_full.parquet
- C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\data\final\publication\idiomx_extended_train.parquet
- C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\data\final\publication\idiomx_extended_test.parquet
